# Dataset de validación y corpus de prueba

Describe los dos insumos de las pruebas: el dataset de preguntas y el documento sobre el que se hicieron.

| Sección | Qué produce | Dónde se usa en el documento |
|---|---|---|
| 1 | Composición del dataset: tipo de evidencia, dificultad, tipo de pregunta, secciones | Tabla 3 y sección 8.2 |
| 2 | Número de páginas del PDF | Pendiente de la sección 8.2 |
| 3 | Fragmentos de texto y de tabla por colección de Qdrant, y longitud de los fragmentos | Pendiente de la sección 8.2 |

Ejecutar desde la carpeta del proyecto. Las secciones 2 y 3 necesitan el PDF y las carpetas locales de Qdrant, y las colecciones **no deben estar abiertas en otro notebook** (Qdrant local bloquea la carpeta). Salidas en `validacion/reporte_corpus/`.

In [1]:
%pip install -q pandas openpyxl pymupdf qdrant-client

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
from pathlib import Path
import pandas as pd

DIR_VALIDACION = Path("./validacion")
RUTA_DATASET = DIR_VALIDACION / "dataset_validacion_final.json"
RUTA_PDF = Path("./documento.pdf")
DIR_SALIDA = DIR_VALIDACION / "reporte_corpus"
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

# Carpetas locales de Qdrant que se usaron en las pruebas
CARPETAS_QDRANT = {
    "base (9 configuraciones)": Path("./qdrant_data"),
    "híbrido BGE-m3": Path("./qdrant_data_hibrido"),
    "BGE-m3 híbrido (tamaño 800)": Path("./qdrant_data_bge_hibrido"),
    "pipeline mejorado": Path("./qdrant_data_mejorado"),
    "producción (configuraciones finales)": Path("./qdrant_data_produccion"),
}

## 1. Composición del dataset (Tabla 3)

In [3]:
contenido = json.load(open(RUTA_DATASET, encoding="utf-8"))
pares = pd.DataFrame(contenido["pares"])
print("Documento fuente:", contenido["documento"], "| pares declarados:", contenido["total_pares"], "| pares leídos:", len(pares))
assert len(pares) == contenido["total_pares"]
print("Secciones de origen distintas:", pares["seccion"].nunique())

filas = []
for atributo, columna in [("Tipo de evidencia", "tipo_evidencia"), ("Dificultad", "dificultad"), ("Tipo de pregunta", "tipo")]:
    conteo = pares[columna].value_counts()
    for categoria, cantidad in conteo.items():
        filas.append({"Atributo": atributo, "Categoría": categoria, "Preguntas": int(cantidad), "%": 100 * cantidad / len(pares)})
tabla3 = pd.DataFrame(filas)
display(tabla3.round(1))

cruce_dif = pd.crosstab(pares["tipo_evidencia"], pares["dificultad"], margins=True)
cruce_tipo = pd.crosstab(pares["tipo"], pares["tipo_evidencia"], margins=True)
display(cruce_dif)
display(cruce_tipo)
por_seccion = pares.groupby("seccion").size().rename("Preguntas").reset_index()
display(por_seccion)

Documento fuente: Acreditacion2024 | pares declarados: 296 | pares leídos: 296
Secciones de origen distintas: 15


,Atributo,Categoría,Preguntas,%
0,Tipo de evidencia,texto,165,55.7
1,Tipo de evidencia,tabla,127,42.9
2,Tipo de evidencia,texto+tabla,4,1.4
3,Dificultad,baja,142,48.0
4,Dificultad,media,131,44.3
5,Dificultad,alta,23,7.8
6,Tipo de pregunta,tabular,117,39.5
7,Tipo de pregunta,factual,67,22.6
8,Tipo de pregunta,descriptiva,57,19.3
9,Tipo de pregunta,multicontexto,18,6.1


dificultad,alta,baja,media,All
tipo_evidencia,,,,
tabla,3,96,28,127
texto,19,46,100,165
texto+tabla,1,0,3,4
All,23,142,131,296


tipo_evidencia,tabla,texto,texto+tabla,All
tipo,,,,
comparativa,8,1,2,11
descriptiva,0,57,0,57
evidencial,0,9,0,9
factual,1,66,0,67
multicontexto,3,13,2,18
negativa / verificación,0,2,0,2
procedimental,0,15,0,15
tabular,115,2,0,117
All,127,165,4,296


,seccion,Preguntas
0,4-informacion-del-programa.pdf,20
1,6-apreciacion-global.pdf,19
2,factor--1.pdf,20
3,factor--10.pdf,20
4,factor--11.pdf,20
5,factor--12.pdf,20
6,factor--3.pdf,20
7,factor--4.pdf,20
8,factor--5.pdf,19
9,factor--6.pdf,20


## 2. Páginas del documento

El PDF usado en las pruebas es `documento.pdf` (el mismo de `archivo_fuente/7RAAC_Ingenierias Sistemas UV-24042025.pdf`).

In [4]:
import pymupdf

with pymupdf.open(RUTA_PDF) as pdf:
    n_paginas = pdf.page_count
print(f"Páginas de {RUTA_PDF.name}: {n_paginas}")

Páginas de documento.pdf: 330


## 3. Fragmentos por colección de Qdrant

Recorre cada colección y cuenta los fragmentos de texto y de tabla, con la longitud de sus contenidos. Los fragmentos de tabla son los mismos para todos los tamaños de chunk de una misma extracción; el texto cambia con el tamaño.

In [5]:
from qdrant_client import QdrantClient

filas = []
for etiqueta, carpeta in CARPETAS_QDRANT.items():
    if not carpeta.exists():
        print("No existe:", carpeta); continue
    cliente = QdrantClient(path=str(carpeta))
    try:
        for col in cliente.get_collections().collections:
            tipos, largos = [], []
            siguiente = None
            while True:
                puntos, siguiente = cliente.scroll(col.name, limit=500, offset=siguiente, with_payload=True, with_vectors=False)
                for p in puntos:
                    tipos.append(p.payload.get("tipo", "sin tipo"))
                    largos.append(len(p.payload.get("page_content", "")))
                if siguiente is None:
                    break
            d = pd.DataFrame({"tipo": tipos, "largo": largos})
            texto, tabla = d[d["tipo"] == "texto"]["largo"], d[d["tipo"] == "tabla"]["largo"]
            filas.append({"Carpeta": etiqueta, "Colección": col.name, "Fragmentos": len(d),
                          "De texto": len(texto), "De tabla": len(tabla),
                          "Texto: largo medio (car.)": texto.mean(), "Tabla: largo medio (car.)": tabla.mean(),
                          "Tablas > 3000 car.": int((tabla > 3000).sum()), "Tabla más larga (car.)": tabla.max() if len(tabla) else None})
    finally:
        cliente.close()
tabla_colecciones = pd.DataFrame(filas)
display(tabla_colecciones.round(1))

,Carpeta,Colección,Fragmentos,De texto,De tabla,Texto: largo medio (car.),Tabla: largo medio (car.),Tablas > 3000 car.,Tabla más larga (car.)
0,base (9 configuraciones),rag_openai_text_embedding_3_large_cs300,973,751,222,861.7,764.5,5,6795.0
1,base (9 configuraciones),rag_openai_text_embedding_3_large_cs500,638,416,222,1558.5,764.5,5,6795.0
2,base (9 configuraciones),rag_openai_text_embedding_3_large_cs800,467,245,222,2649.1,764.5,5,6795.0
3,base (9 configuraciones),rag_baai_bge_m3_cs300,831,609,222,1063.4,764.5,5,6795.0
4,base (9 configuraciones),rag_baai_bge_m3_cs500,560,338,222,1919.1,764.5,5,6795.0
5,base (9 configuraciones),rag_baai_bge_m3_cs800,422,200,222,3245.8,764.5,5,6795.0
6,base (9 configuraciones),rag_google_embeddinggemma_300m_cs300,844,622,222,1041.1,764.5,5,6795.0
7,base (9 configuraciones),rag_google_embeddinggemma_300m_cs500,572,350,222,1853.2,764.5,5,6795.0
8,base (9 configuraciones),rag_google_embeddinggemma_300m_cs800,430,208,222,3120.8,764.5,5,6795.0
9,híbrido BGE-m3,rag_baai_bge_m3_hibrido_cs500,0,0,0,NaN,NaN,0,NaN


## 4. Exportar

In [6]:
with pd.ExcelWriter(DIR_SALIDA / "reporte_corpus.xlsx") as xw:
    tabla3.to_excel(xw, sheet_name="Tabla3_dataset", index=False)
    cruce_dif.to_excel(xw, sheet_name="Evidencia_x_dificultad")
    cruce_tipo.to_excel(xw, sheet_name="Tipo_x_evidencia")
    por_seccion.to_excel(xw, sheet_name="Por_seccion", index=False)
    tabla_colecciones.to_excel(xw, sheet_name="Colecciones_Qdrant", index=False)
tabla3.to_csv(DIR_SALIDA / "tabla3_dataset.csv", index=False, encoding="utf-8-sig")
tabla_colecciones.to_csv(DIR_SALIDA / "colecciones_qdrant.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"Concepto": ["Páginas del PDF"], "Valor": [n_paginas]}).to_csv(DIR_SALIDA / "pdf.csv", index=False, encoding="utf-8-sig")
print("Guardado en", DIR_SALIDA.resolve())

Guardado en C:\Users\gomex\OneDrive\Escritorio\TG - Pruebas 2\validacion\reporte_corpus
